## 10-statement-linreg

Ответ: 56572.23 37191.87

In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from scipy.sparse import hstack
from sklearn.linear_model import Ridge

# ------------------------------------------------------------
# 1. Загрузка данных
# ------------------------------------------------------------
train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')

# Целевая переменная
y_train = train['SalaryNormalized']

# ------------------------------------------------------------
# 2. Предобработка текстов
# ------------------------------------------------------------
def text_preprocess(text):
    # приведение к нижнему регистру, замена всего кроме букв и цифр на пробел
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    return text

train['FullDescription'] = train['FullDescription'].apply(text_preprocess)
test['FullDescription'] = test['FullDescription'].apply(text_preprocess)

# ------------------------------------------------------------
# 3. TF-IDF (min_df=5)
# ------------------------------------------------------------
tfidf = TfidfVectorizer(min_df=5)
X_train_text = tfidf.fit_transform(train['FullDescription'])
X_test_text = tfidf.transform(test['FullDescription'])

# ------------------------------------------------------------
# 4. Категориальные признаки: заполнение пропусков и DictVectorizer
# ------------------------------------------------------------
# Заполняем пропуски строкой 'nan'
train['LocationNormalized'] = train['LocationNormalized'].fillna('nan')
train['ContractTime'] = train['ContractTime'].fillna('nan')
test['LocationNormalized'] = test['LocationNormalized'].fillna('nan')
test['ContractTime'] = test['ContractTime'].fillna('nan')

# Преобразуем в список словарей для DictVectorizer
train_dict = train[['LocationNormalized', 'ContractTime']].to_dict('records')
test_dict = test[['LocationNormalized', 'ContractTime']].to_dict('records')

dv = DictVectorizer()
X_train_categ = dv.fit_transform(train_dict)
X_test_categ = dv.transform(test_dict)

# ------------------------------------------------------------
# 5. Объединение признаков
# ------------------------------------------------------------
X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])

# ------------------------------------------------------------
# 6. Обучение гребневой регрессии (alpha=1)
# ------------------------------------------------------------
ridge = Ridge(alpha=1)
ridge.fit(X_train, y_train)

# ------------------------------------------------------------
# 7. Прогноз для двух тестовых примеров
# ------------------------------------------------------------
predictions = ridge.predict(X_test)
# Округляем до двух знаков
predictions_rounded = [round(p, 2) for p in predictions]

# Вывод через пробел
print(f"{predictions_rounded[0]} {predictions_rounded[1]}")

56572.23 37191.87
